In [1]:
pip install requests pandas streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 30.9 MB/s eta 0:00:00


Create analysis.py inside Colab

In [4]:
%%writefile analysis.py

import requests
import pandas as pd

def get_commits(owner, repo):
    url = f"https://api.github.com/repos/{owner}/{repo}/commits"
    response = requests.get(url)
    data = response.json()

    commits = []
    for c in data:
        commit_msg = c["commit"]["message"]
        author = c["commit"]["author"]["name"]

        commits.append({
            "author": author,
            "message": commit_msg
        })

    return pd.DataFrame(commits)


def classify_commit(msg):
    msg = msg.lower()
    if "fix" in msg:
        return "Bug Fix"
    elif "feat" in msg:
        return "Feature"
    else:
        return "Other"


def analyze_repo(owner, repo):
    df = get_commits(owner, repo)

    df["type"] = df["message"].apply(classify_commit)

    contrib = df["author"].value_counts().reset_index()
    contrib.columns = ["author", "commits"]

    total = contrib["commits"].sum()
    contrib["cum_sum"] = contrib["commits"].cumsum()
    bus_factor = (contrib["cum_sum"] < 0.5 * total).sum() + 1

    return df, contrib, bus_factor

Writing analysis.py


Backend Logic

In [5]:
import requests
import pandas as pd

def get_commits(owner, repo):
    url = f"https://api.github.com/repos/{owner}/{repo}/commits"
    response = requests.get(url)
    data = response.json()

    commits = []
    for c in data:
        commit_msg = c["commit"]["message"]
        author = c["commit"]["author"]["name"]

        commits.append({
            "author": author,
            "message": commit_msg
        })

    return pd.DataFrame(commits)


def classify_commit(msg):
    msg = msg.lower()
    if "fix" in msg:
        return "Bug Fix"
    elif "feat" in msg:
        return "Feature"
    else:
        return "Other"


def analyze_repo(owner, repo):
    df = get_commits(owner, repo)

    # Classification
    df["type"] = df["message"].apply(classify_commit)

    # Contribution count
    contrib = df["author"].value_counts().reset_index()
    contrib.columns = ["author", "commits"]

    # Bus factor
    total = contrib["commits"].sum()
    contrib["cum_sum"] = contrib["commits"].cumsum()
    bus_factor = (contrib["cum_sum"] < 0.5 * total).sum() + 1

    return df, contrib, bus_factor

Frontend Dashboard

In [6]:
import streamlit as st
from analysis import analyze_repo

st.title("🚀 Pulse: GitHub Engineering Dashboard")

repo_url = st.text_input("Enter GitHub Repo URL")

if repo_url:
    try:
        parts = repo_url.split("/")
        owner = parts[-2]
        repo = parts[-1]

        df, contrib, bus_factor = analyze_repo(owner, repo)

        st.subheader("📊 Contribution Breakdown")
        st.bar_chart(contrib.set_index("author"))

        st.subheader("📌 Commit Types")
        st.write(df["type"].value_counts())

        st.subheader("🧍 Bus Factor")
        st.write(bus_factor)

        st.subheader("📄 Sample Commits")
        st.write(df.head())

    except Exception as e:
        st.error("Error fetching data")

2026-03-31 13:37:51.178 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 13:37:51.331 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-31 13:37:51.332 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 13:37:51.333 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 13:37:51.336 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 13:37:51.336 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 13:37:51.338 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 13:37:51.340 Thread 'MainThread': mi

In [9]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [10]:
%%writefile app.py

import streamlit as st
from analysis import analyze_repo

st.title("🚀 Pulse: GitHub Engineering Dashboard")

repo_url = st.text_input("Enter GitHub Repo URL")

if repo_url:
    try:
        parts = repo_url.split("/")
        owner = parts[-2]
        repo = parts[-1]

        df, contrib, bus_factor = analyze_repo(owner, repo)

        st.subheader("📊 Contribution Breakdown")
        st.bar_chart(contrib.set_index("author"))

        st.subheader("📌 Commit Types")
        st.write(df["type"].value_counts())

        st.subheader("🧍 Bus Factor")
        st.write(bus_factor)

        st.subheader("📄 Sample Commits")
        st.write(df.head())

    except Exception as e:
        st.error("Error fetching data")

Writing app.py


In [ ]:
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.194.215.160:8501

